#1 Mini datasheet
Dataset Card — UCI HAR
Motivation: Benchmark ML models for passive human activity recognition using smartphone inertial sensors. Downstream applications include health monitoring, elderly care, and wearable computing.
Source & License: UCI Machine Learning Repository, Reyes-Ortiz et al. (2013). Public domain for academic use; commercial use prohibited. Cite Anguita et al., ESANN 2013.
Participants: 30 volunteers, ages 19–48, Samsung Galaxy S II worn on the waist. Lab sessions were video-recorded for manual annotation.
Target: Six activity classes — WALKING, WALKING_UPSTAIRS, WALKING_DOWNSTAIRS, SITTING, STANDING, LAYING.
Signals & Features: 3-axis accelerometer + 3-axis gyroscope at 50 Hz. Windows of 2.56 s with 50% overlap (128 readings). Body/gravity separation and FFT applied; 561 statistical features extracted per window. All features normalised to [−1, 1]. This partition: 2,947 windows × 561 features across 9 subjects.
Limitations & Risks:

Lab bias — activities performed on demand; real-world motion is noisier and more ambiguous
Sensor specificity — single phone model, waist placement; may not generalise to other devices or positions
Demographic gaps — 30 subjects; age, gait, and body diversity are uncharacterised
Coarse labels — 6 static activities; transitions and fine-grained states are unlabelled
Leakage risk — 50% window overlap means random row splits leak near-duplicate windows across folds; subject-grouped splitting is required


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#2 Data sanity check
import numpy as np
import os

# Data Sanity Checks
def load_split(split_path):
    signals = [
        "body_acc_x", "body_acc_y", "body_acc_z",
        "body_gyro_x", "body_gyro_y", "body_gyro_z",
        "total_acc_x", "total_acc_y", "total_acc_z"
    ]

    data_list = []

    for sig in signals:
        file_path = os.path.join(split_path, f"{sig}_{os.path.basename(split_path)}.txt")
        data = np.loadtxt(file_path)
        data_list.append(data)

    # Stack as (num_windows, T, C)
    X = np.stack(data_list, axis=-1)
    return X
X_train = load_split("train")
X_test = load_split("test")

# Shape Check
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Expected format: (num_windows, T, C)")


# NaN / Inf Check
def check_nan_inf(X, name):
    print(f"\nChecking {name}")
    print("Contains NaN:", np.isnan(X).any())
    print("Contains Inf:", np.isinf(X).any())

check_nan_inf(X_train, "Train")
check_nan_inf(X_test, "Test")

# Value Range Check
def check_range(X, name):
    print(f"\n{name} value range:")
    print("Min:", X.min())
    print("Max:", X.max())
    print("Mean:", X.mean())
    print("Std:", X.std())

# Label Distribution Check
check_range(X_train, "Train")
check_range(X_test, "Test")

y_train = np.loadtxt("train/y_train.txt")
y_test = np.loadtxt("test/y_test.txt")

print("Train label distribution:")
unique, counts = np.unique(y_train, return_counts=True)
print(dict(zip(unique.astype(int), counts)))

print("\nTest label distribution:")
unique, counts = np.unique(y_test, return_counts=True)
print(dict(zip(unique.astype(int), counts)))




Train shape: (7352, 128, 9)
Test shape: (2947, 128, 9)
Expected format: (num_windows, T, C)

Checking Train
Contains NaN: False
Contains Inf: False

Checking Test
Contains NaN: False
Contains Inf: False

Train value range:
Min: -5.97433
Max: 5.746062
Mean: 0.10206605723804434
Std: 0.4021651763827932

Test value range:
Min: -3.431701
Max: 3.468111
Mean: 0.09913989069610847
Std: 0.39567084061541463
Train label distribution:
{np.int64(1): np.int64(1226), np.int64(2): np.int64(1073), np.int64(3): np.int64(986), np.int64(4): np.int64(1286), np.int64(5): np.int64(1374), np.int64(6): np.int64(1407)}

Test label distribution:
{np.int64(1): np.int64(496), np.int64(2): np.int64(471), np.int64(3): np.int64(420), np.int64(4): np.int64(491), np.int64(5): np.int64(532), np.int64(6): np.int64(537)}


In [ ]:
#3 Leakage audit

In [ ]:
import numpy as np
import os

base_path = "/content"

In [ ]:
train_files = [
    "body_acc_x_train.txt",
    "body_acc_y_train.txt",
    "body_acc_z_train.txt",
    "body_gyro_x_train.txt",
    "body_gyro_y_train.txt",
    "body_gyro_z_train.txt",
    "total_acc_x_train.txt",
    "total_acc_y_train.txt",
    "total_acc_z_train.txt"
]

X_train_list = []

for file in train_files:
    data = np.loadtxt(os.path.join(base_path, file))
    X_train_list.append(data)

# Stack as channels
X_train = np.stack(X_train_list, axis=-1)

print("X_train shape:", X_train.shape)


X_train shape: (7352, 128, 9)


In [ ]:
test_files = [
    "body_acc_x_test.txt",
    "body_acc_y_test.txt",
    "body_acc_z_test.txt",
    "body_gyro_x_test.txt",
    "body_gyro_y_test.txt",
    "body_gyro_z_test.txt",
    "total_acc_x_test.txt",
    "total_acc_y_test.txt",
    "total_acc_z_test.txt"
]

X_test_list = []

for file in test_files:
    data = np.loadtxt(os.path.join(base_path, file))
    X_test_list.append(data)

X_test = np.stack(X_test_list, axis=-1)

print("X_test shape:", X_test.shape)


X_test shape: (2947, 128, 9)


In [ ]:
subject_train = np.loadtxt(os.path.join(base_path, "subject_train.txt")).astype(int)
subject_test  = np.loadtxt(os.path.join(base_path, "subject_test.txt")).astype(int)

print("subject_train shape:", subject_train.shape)
print("subject_test shape:", subject_test.shape)

subject_train shape: (7352,)
subject_test shape: (2947,)


In [ ]:
#leakage audit = proving that no subject appears in both train and test
train_subjects = set(subject_train)
test_subjects  = set(subject_test)

overlap = train_subjects & test_subjects

print("Train subjects:", sorted(train_subjects))
print("Test subjects:", sorted(test_subjects))
print("Overlap:", overlap)
print("Overlap size:", len(overlap))


Train subjects: [np.int64(1), np.int64(3), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(11), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(19), np.int64(21), np.int64(22), np.int64(23), np.int64(25), np.int64(26), np.int64(27), np.int64(28), np.int64(29), np.int64(30)]
Test subjects: [np.int64(2), np.int64(4), np.int64(9), np.int64(10), np.int64(12), np.int64(13), np.int64(18), np.int64(20), np.int64(24)]
Overlap: set()
Overlap size: 0


In [ ]:
#4 Baseline 0
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, classification_report

def load_split(split_path):
    signals = [
        "body_acc_x", "body_acc_y", "body_acc_z",
        "body_gyro_x", "body_gyro_y", "body_gyro_z",
        "total_acc_x", "total_acc_y", "total_acc_z"
    ]

    data_list = []

    for sig in signals:
        file_path = os.path.join(split_path, f"{sig}_{os.path.basename(split_path)}.txt")
        data = np.loadtxt(file_path)
        data_list.append(data)

    X = np.stack(data_list, axis=-1)
    return X

X_train = load_split("train")
X_test = load_split("test")


baseline_majority = DummyClassifier(strategy="most_frequent")
baseline_majority.fit(X_train, y_train)
y_pred_maj = baseline_majority.predict(X_test)

baseline_random = DummyClassifier(strategy="stratified", random_state=42)
baseline_random.fit(X_train, y_train)
y_pred_rand = baseline_random.predict(X_test)

acc_maj = accuracy_score(y_test, y_pred_maj)
acc_rand = accuracy_score(y_test, y_pred_rand)

print(f"Baseline 0 - Majority Class Accuracy: {acc_maj:.4f}")
print(f"Baseline 0 - Random (Seed 42) Accuracy: {acc_rand:.4f}")

Baseline 0 - Majority Class Accuracy: 0.1822
Baseline 0 - Random (Seed 42) Accuracy: 0.1710


In [ ]:
#5 Baseline 1
import numpy as np
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
#MLP on flattened windows (T × C) with a train/validation split drawn only from training subjects
y_train = np.loadtxt(os.path.join(base_path, "y_train.txt")).astype(int)
y_test  = np.loadtxt(os.path.join(base_path, "y_test.txt")).astype(int)
x_train = np.loadtxt(os.path.join(base_path, "X_train.txt")).astype(int)
x_test  = np.loadtxt(os.path.join(base_path, "X_test.txt")).astype(int)

X_train_flat = X_train.reshape(len(X_train), -1)
X_test_flat  = X_test.reshape(len(X_test), -1)

print("X_train_flat:", X_train_flat.shape)  # (7352, 1152)

# Subject-disjoint train/val split (ONLY from training subjects)
unique_subjects = np.unique(subject_train)

train_subj, val_subj = train_test_split(
    unique_subjects,
    test_size=0.2,
    random_state=42
)

train_mask = np.isin(subject_train, train_subj)
val_mask   = np.isin(subject_train, val_subj)

X_tr, y_tr = X_train_flat[train_mask], y_train[train_mask]
X_val, y_val = X_train_flat[val_mask], y_train[val_mask]

print("Train windows:", X_tr.shape, "Val windows:", X_val.shape)

#  Scale features (fit on train only)
scaler = StandardScaler()
X_tr_s  = scaler.fit_transform(X_tr)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test_flat)

# Train MLP baseline

mlp = MLPClassifier(
    hidden_layer_sizes=(256, 128),
    activation="relu",
    solver="adam",
    alpha=1e-4,
    batch_size=256,
    learning_rate_init=1e-3,
    max_iter=50,
    random_state=42,
    early_stopping=True,
    n_iter_no_change=5,
    validation_fraction=0.1
)

mlp.fit(X_tr_s, y_tr)

val_pred = mlp.predict(X_val_s)
test_pred = mlp.predict(X_test_s)

print("\nVAL accuracy:", accuracy_score(y_val, val_pred))
print("VAL macro-F1:", f1_score(y_val, val_pred, average="macro"))

print("\nTEST accuracy:", accuracy_score(y_test, test_pred))
print("TEST macro-F1:", f1_score(y_test, test_pred, average="macro"))

print("\nConfusion matrix (test):\n", confusion_matrix(y_test, test_pred))
print("\nPer-class report (test):\n", classification_report(y_test, test_pred))
#The MLP baseline achieved 88.0% accuracy (macro-F1 = 0.881) on subject-disjoint test data.
#Performance is strong for dynamic activities (F1 ≈ 0.90), while confusion is primarily observed between sitting
#and standing due to similar static inertial patterns. Laying is classified with high accuracy (F1 = 0.97),
#likely due to distinct device orientation features.

X_train_flat: (7352, 1152)
Train windows: (5551, 1152) Val windows: (1801, 1152)

VAL accuracy: 0.9344808439755691
VAL macro-F1: 0.9348994536374273

TEST accuracy: 0.8802171700033933
TEST macro-F1: 0.8814493810158787

Confusion matrix (test):
 [[448  10  21  12   5   0]
 [ 16 427  13   4  11   0]
 [  3  20 372   6  19   0]
 [  5   3   1 402  80   0]
 [  1   2   1  93 435   0]
 [  0  27   0   0   0 510]]

Per-class report (test):
               precision    recall  f1-score   support

           1       0.95      0.90      0.92       496
           2       0.87      0.91      0.89       471
           3       0.91      0.89      0.90       420
           4       0.78      0.82      0.80       491
           5       0.79      0.82      0.80       532
           6       1.00      0.95      0.97       537

    accuracy                           0.88      2947
   macro avg       0.88      0.88      0.88      2947
weighted avg       0.88      0.88      0.88      2947



In [ ]:
#6 Metric